### Making the dataset file structure compatible with the SDG Engine's

Convert the TUDL dataset to an easily loadable format for Hugging Face's datasets library.

In [24]:
import os
from typing import List, Dict
import shutil
import json

DATASET_SPLIT = "train_real"
ROOT_FOLDER = "/Users/federico/Documents/personal/projects/code/vit-sdg-engine"
DATASET_LOCATION = f"{ROOT_FOLDER}/tudl-dataset"

dataset_jsonl: List[Dict] = []
dataset_path = f"{DATASET_LOCATION}/{DATASET_SPLIT}"
target_dataset_path = f"{ROOT_FOLDER}/hf-tudl-rgb-dataset/train"

# if the target dataset path does not exist, create it
if not os.path.exists(target_dataset_path):
    os.makedirs(target_dataset_path)

for folder in os.listdir(dataset_path):
    if os.path.isdir(os.path.join(dataset_path, folder)):
        rgb_images = os.listdir(os.path.join(dataset_path, folder, "rgb"))

        # load the scene_gt_info.json file
        scene_gt_info = json.load(
            open(os.path.join(dataset_path, folder, "scene_gt_info.json"))
        )
        object_class = folder.lstrip("0")
        for image in rgb_images:
            annotation_dict = {}
            # Copy the image to the target dataset path
            img_new_path = os.path.join(target_dataset_path, image)
            shutil.copy(
                os.path.join(dataset_path, folder, "rgb", image),
                img_new_path,
            )
            # Collect the object's bounding box from the scene_gt_info.json file
            image_id = image.split(".")[0].lstrip("0") or "0"
            if len(scene_gt_info[image_id]) == 1:
                image_bbox = scene_gt_info[image_id][0]["bbox_obj"]
            else:
                raise ValueError(
                    f"Expected 1 object in the scene_gt_info.json file for image {image}, but got {len(scene_gt_info[image_id])}"
                )
            # Populate the annotation dictionary
            annotation_dict["image_id"] = image_id
            annotation_dict["file_name"] = img_new_path
            annotation_dict["objects"] = {
                "bbox": [image_bbox],
                "categories": [[int(object_class)]],
            }

            dataset_jsonl.append(annotation_dict)

# Finally, save the dataset_jsonl to a file
with open(f"{target_dataset_path}/metadata.jsonl", "w") as f:
    for annotation in dataset_jsonl:
        f.write(json.dumps(annotation) + "\n")

Now load them into Hugging Face datasets.

In [25]:
from datasets import load_dataset

DATASET_PATH = "/Users/federico/Documents/personal/projects/code/vit-sdg-engine/hf-tudl-rgb-dataset/"
dataset = load_dataset("imagefolder", data_dir=DATASET_PATH)
print(f"Dataset loaded: \n{dataset}")

Generating train split: 38288 examples [00:04, 8990.93 examples/s] 
Generating test split: 600 examples [00:00, 9941.35 examples/s]


Dataset loaded: 
DatasetDict({
    train: Dataset({
        features: ['image_id', 'image', 'objects'],
        num_rows: 38288
    })
    test: Dataset({
        features: ['image_id', 'image', 'objects'],
        num_rows: 600
    })
})


In [22]:
DATASET_PATH = "/Users/federico/Documents/personal/projects/code/vit-sdg-engine/hf-tudl-rgb-dataset-render/"
dataset_render = load_dataset("imagefolder", data_dir=DATASET_PATH)
print(f"Dataset loaded: \n{dataset_render}")

Generating train split: 5481 examples [00:00, 10415.32 examples/s]


Dataset loaded: 
DatasetDict({
    train: Dataset({
        features: ['image_id', 'image', 'objects'],
        num_rows: 5481
    })
})


Finally, lets push them to the Hugging Face Hub.

In [26]:
HF_TOKEN = os.environ.get("HF_TOKEN")

dataset.push_to_hub(
    "federicoarenas-ai/bop-tudl-rgb-dataset-real",
    token=HF_TOKEN,
)

Uploading the dataset shards: 100%|██████████| 1/1 [00:13<00:00, 13.84s/it]


CommitInfo(commit_url='https://huggingface.co/datasets/federicoarenas-ai/bop-tudl-rgb-dataset-real/commit/86ef245416f9afa5307b7b98299ec63a3ba275f9', commit_message='Upload dataset', commit_description='', oid='86ef245416f9afa5307b7b98299ec63a3ba275f9', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/federicoarenas-ai/bop-tudl-rgb-dataset-real', endpoint='https://huggingface.co', repo_type='dataset', repo_id='federicoarenas-ai/bop-tudl-rgb-dataset-real'), pr_revision=None, pr_num=None)

In [ ]:
dataset_render.push_to_hub(
    "federicoarenas-ai/bop-tudl-rgb-dataset-render",
    token=HF_TOKEN,
)

Uploading the dataset shards: 100%|██████████| 1/1 [00:09<00:00,  9.42s/it]


CommitInfo(commit_url='https://huggingface.co/datasets/federicoarenas-ai/bop-tudl-rgb-dataset-render/commit/e88d060fbdd3e403b42c2917c374a7ce5562920b', commit_message='Upload dataset', commit_description='', oid='e88d060fbdd3e403b42c2917c374a7ce5562920b', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/federicoarenas-ai/bop-tudl-rgb-dataset-render', endpoint='https://huggingface.co', repo_type='dataset', repo_id='federicoarenas-ai/bop-tudl-rgb-dataset-render'), pr_revision=None, pr_num=None)